# Notebook 03 — Avaliação de SLOs, Error Budgets e Observabilidade

## Objetivos
1. Avaliar SLOs e calcular error budgets conforme Beyer et al. (2016).
2. Construir dashboard de SLOs inspirado na Figura 2 do material.
3. Analisar métricas por modalidade e error budgets restantes.
4. Demonstrar logging estruturado para observabilidade (Sridharan, 2018).
5. Discutir technical debt em ML (Sculley et al., 2015).

## Conceito-chave (Documento 04)
> "O conceito de Error Budget emerge naturalmente da definição de SLOs. Se um sistema
> possui um SLO de 99.9% de disponibilidade, o error budget correspondente é de 0.1%,
> ou aproximadamente 43 minutos de downtime permitido por mês."
> — Seção "Fundamentos de SRE para Machine Learning"

> "Observabilidade é um conceito que transcende o monitoramento tradicional, englobando
> a capacidade de compreender o estado interno de um sistema a partir de suas saídas
> externas. O modelo de três pilares da observabilidade [Sridharan, 2018] estabelece
> métricas, logs e traces como os componentes fundamentais."
> — Seção "Monitoramento de Latência e Observabilidade em Produção"

In [ ]:
# Imports
import sys, os
sys.path.insert(0, os.path.abspath(".."))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from src.data_preprocessing import DataPreprocessor
from src.model import SLOMonitor, CanaryAnalyzer
from src.evaluation import (
    calculate_metrics, calculate_error_budget,
    plot_latency_distribution, plot_error_budget_gauge,
    plot_canary_comparison, plot_slo_dashboard,
)
from src.utils import save_metrics, setup_logger, log_inference_event

sns.set_theme(style="whitegrid")
%matplotlib inline

## 1. Carregar e Preparar Dados

In [ ]:
dp = DataPreprocessor(seed=42)
data_path = os.path.join("..", "data", "raw", "dataset.csv")
df = dp.load_data(data_path)
df = dp.clean_data(df)
df = dp.prepare_features(df)

# Split cronológico (últimas 20% = "período de avaliação")
df_train, df_test = dp.split_data(df, test_size=0.2)
print(f"Treino: {len(df_train)} | Teste: {len(df_test)}")

## 2. Cálculo Completo de Métricas SRE

Implementa o cálculo abrangente de SLIs, SLOs e métricas por modalidade/versão
conforme o framework descrito na seção *Fundamentos de SRE para Machine Learning*.

In [ ]:
metrics = calculate_metrics(df, slo_latency_p99_ms=200.0, slo_availability=0.999)

print("=== SLIs ===")
for k, v in metrics["slis"].items():
    print(f"  {k:20s}: {v:.4f}")

print("\n=== SLOs ===")
for slo in metrics["slos"]:
    status = "✓" if slo["met"] else "✗"
    print(f"  [{status}] {slo['name']:20s}: target={slo['target']}, "
          f"actual={slo['actual']}, budget_remaining={slo['error_budget_remaining_pct']}%")

## 3. Error Budget Detalhado

O error budget transforma confiabilidade de objetivo binário em recurso gerenciável
(Beyer et al., 2016). Calculamos quanto do budget foi consumido no período.

In [ ]:
budget = calculate_error_budget(
    total_requests=len(df),
    error_count=int(df["is_error"].sum()),
    slo_availability=0.999,
    window_days=30,
)

print("=== Error Budget ===")
for k, v in budget.items():
    print(f"  {k:30s}: {v}")

In [ ]:
# Gauge de error budget
fig = plot_error_budget_gauge(
    budget["budget_remaining_pct"],
    slo_name="Disponibilidade 99.9%",
    save_path=os.path.join("..", "outputs", "figures", "error_budget_gauge.png"),
)
plt.show()

## 4. Dashboard de SLOs

Inspirado na Figura 2 do material (*Dashboard de Monitoramento de SLOs
para Inferência de ML*), o dashboard consolida:
- Percentis de latência com linha de SLO.
- Disponibilidade vs meta.
- P99 por modalidade.
- Error budget consumido.

In [ ]:
fig = plot_slo_dashboard(
    metrics,
    save_path=os.path.join("..", "outputs", "figures", "slo_dashboard.png"),
)
plt.show()

## 5. Distribuição de Latência com SLO

In [ ]:
fig = plot_latency_distribution(
    df,
    slo_target_ms=200.0,
    save_path=os.path.join("..", "outputs", "figures", "latency_distribution.png"),
)
plt.show()

## 6. Métricas por Modalidade

Conforme a arquitetura multimodal (Figura 1), cada worker tem perfil
de latência e taxa de erro distintos.

In [ ]:
mod_df = pd.DataFrame(metrics["per_modality"]).T
mod_df.index.name = "modality"
print("Métricas por Modalidade:")
mod_df.round(4)

In [ ]:
# Comparação visual
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

mod_df["latency_p99"].plot.bar(ax=axes[0], color=["#3498db", "#e67e22", "#9b59b6"])
axes[0].axhline(200, color="red", linestyle="--", label="SLO")
axes[0].set_title("Latência P99 por Modalidade")
axes[0].set_ylabel("ms")
axes[0].legend()

mod_df["error_rate"].plot.bar(ax=axes[1], color=["#3498db", "#e67e22", "#9b59b6"])
axes[1].set_title("Taxa de Erro por Modalidade")
axes[1].set_ylabel("Taxa")

plt.tight_layout()
plt.show()

## 7. Análise Canary — Testes Estatísticos

Aplicamos teste-t (latência) e qui-quadrado (taxa de erro) conforme
a Tabela 2 do material (*Framework de Decisão para Canary Deployment*).

Fórmula do teste-t:
$$t = \frac{\bar{X}_1 - \bar{X}_2}{\sqrt{\frac{s_1^2}{n_1} + \frac{s_2^2}{n_2}}}$$

In [ ]:
analyzer = CanaryAnalyzer(alpha=0.05)
results = analyzer.full_analysis(df)

print("=== Análise Canary ===")
for name, res in results.items():
    print(f"\n{name}:")
    print(f"  Teste: {res.test_name}")
    print(f"  Estatística: {res.statistic:.4f}")
    print(f"  p-valor: {res.p_value:.6f}")
    print(f"  Significativo (α=0.05): {res.significant}")
    print(f"  Decisão: {res.decision.upper()}")

In [ ]:
# Stable vs Canary — métricas descritivas
ver_df = pd.DataFrame(metrics["per_version"]).T
ver_df.index.name = "version"
ver_df.round(4)

## 8. Logging Estruturado — Observabilidade

Conforme Sridharan (2018), logs estruturados em JSON com campos padronizados
permitem análise automatizada. Demonstramos o padrão recomendado na seção
*Monitoramento de Latência e Observabilidade em Produção*.

Campos recomendados: `timestamp`, `request_id`, `model_version`,
`inference_time_ms`, `prediction_confidence`.

In [ ]:
import logging

# Configurar logger JSON
logger = setup_logger(
    name="notebook_demo",
    log_file=os.path.join("..", "outputs", "logs", "inference.log"),
)

# Simular alguns eventos de inferência
sample = df.sample(5, random_state=42)
for _, row in sample.iterrows():
    log_inference_event(
        logger=logger,
        request_id=row["request_id"],
        model_version=row["model_version"],
        modality=row["modality"],
        latency_ms=row["total_latency_ms"],
        is_error=bool(row["is_error"]),
        confidence=row["confidence"],
    )

# Remover handlers para não duplicar em re-execuções
logger.handlers.clear()
print("Eventos logados em outputs/logs/inference.log")

## 9. Technical Debt em ML — Análise Qualitativa

Sculley et al. (2015) identificaram que < 5 % de um sistema de ML
é código de modelo. O restante é infraestrutura:

| Componente | % Estimado |
|-----------|------------|
| Código de modelo (training + inference) | ~5% |
| Ingestão e validação de dados | ~20% |
| Feature engineering | ~15% |
| Configuração e infra | ~25% |
| Serving e API | ~15% |
| Monitoramento e alertas | ~10% |
| Testes e CI/CD | ~10% |

Nesta aula, cobrimos serving (~15%), monitoramento (~10%) e configuração (~25%),
que juntos representam **metade** do sistema total.

In [ ]:
# Visualizar distribuição de componentes de um sistema ML
components = {
    "Modelo (train+infer)": 5,
    "Dados (ingestão)": 20,
    "Feature Engineering": 15,
    "Configuração/Infra": 25,
    "Serving/API": 15,
    "Monitoramento": 10,
    "Testes/CI-CD": 10,
}

fig, ax = plt.subplots(figsize=(8, 8))
colors = ["#e74c3c", "#3498db", "#2ecc71", "#f39c12", "#9b59b6", "#1abc9c", "#34495e"]
wedges, texts, autotexts = ax.pie(
    components.values(), labels=components.keys(), autopct="%1.0f%%",
    colors=colors, startangle=90, pctdistance=0.85
)
ax.set_title("Distribuição de Código em Sistemas de ML\n(Sculley et al., 2015)",
             fontsize=13)
plt.tight_layout()
plt.savefig(os.path.join("..", "outputs", "figures", "tech_debt_pie.png"),
            dpi=150, bbox_inches="tight")
plt.show()

## 10. Salvar Métricas Finais

In [ ]:
# Persistir métricas em JSON
save_metrics(
    metrics,
    os.path.join("..", "outputs", "logs", "slo_metrics.json"),
)
save_metrics(
    budget,
    os.path.join("..", "outputs", "logs", "error_budget.json"),
)
print("Métricas salvas em outputs/logs/")

## Resumo

Neste notebook:
- Avaliamos SLOs de latência P99 e disponibilidade, com error budgets.
- Construímos dashboard de SLOs inspirado na Figura 2 do material.
- Analisamos métricas por modalidade (text, image, audio).
- Executamos testes estatísticos (t-test, χ²) para canary deployment.
- Demonstramos logging estruturado para observabilidade.
- Discutimos technical debt em ML (Sculley et al., 2015).

### Referências do Material
- Beyer, B. et al. (2016). *Site Reliability Engineering*. O'Reilly.
- Sculley, D. et al. (2015). *Hidden Technical Debt in ML Systems*. NeurIPS.
- Sridharan, C. (2018). *Distributed Systems Observability*. O'Reilly.
- Paleyes, A. et al. (2022). *Challenges in Deploying ML*. ACM Comp. Surveys.